In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import os
import re

In [2]:
print(" PASTRIMI I TË DHËNAVE PËR MAKINA")

DATA_DIR = Path('data')
CLEANED_DIR = DATA_DIR / 'cleaned'
REPORTS_DIR = DATA_DIR / 'reports'

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n Folderi i të dhënave: {DATA_DIR}")
print(f" Folderi i pastruar: {CLEANED_DIR}")
print(f" Folderi i raporteve: {REPORTS_DIR}")

 PASTRIMI I TË DHËNAVE PËR MAKINA

 Folderi i të dhënave: data
 Folderi i pastruar: data\cleaned
 Folderi i raporteve: data\reports


In [3]:
def convert_price_to_numeric(price_series):
    
    def clean_price(value):
        if pd.isna(value):
            return np.nan
        value_str = str(value)
        
        cleaned = re.sub(r'[^\d.]', '', value_str)
        if cleaned == '' or cleaned == '.':
            return np.nan
        try:
            return float(cleaned)
        except:
            return np.nan
    return price_series.apply(clean_price)



def clean_car_dataset(brand, display_name, df):
   
    
    original_rows = len(df)
    print(f"\n    Pastrimi i {brand}...")
    

    df = df.drop_duplicates()
    
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
   
    if 'price' in df.columns:
        print(f"    Konvertimi i kolonës 'price'...")
        df['price'] = convert_price_to_numeric(df['price'])
        print(f"       Tipi i ri: {df['price'].dtype}")
    
    
    numeric_cols = ['year', 'mileage', 'engine_size', 'horsepower', 'mpg']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
   
    important_cols = [col for col in df.columns if col in ['price', 'year', 'mileage']]
    if important_cols:
        before = len(df)
        df = df.dropna(subset=important_cols)
        print(f"   Hequr vlera null: {before - len(df)} rreshta")
    
   
    if 'unclean' in brand:
        print(f"    Pastrim i thellë për të dhënat e papastra...")
        
        if 'price' in df.columns and len(df) > 0:
            median_price = df['price'].median()
            if not pd.isna(median_price):
                before = df['price'].isna().sum()
                df['price'] = df['price'].fillna(median_price)
                print(f"       Plotësuar {before} vlera null të çmimit me medianën: {median_price}")
        
        if 'year' in df.columns and len(df) > 0:
            before = df['year'].isna().sum()
            df['year'] = df['year'].fillna(2000).astype(int)
            print(f"       Plotësuar {before} vlera null të vitit me 2000")
        
        if 'mileage' in df.columns and len(df) > 0:
            before = df['mileage'].isna().sum()
            df['mileage'] = df['mileage'].fillna(0)
            print(f"       Plotësuar {before} vlera null të kilometrazhit me 0")

    if 'price' in df.columns and len(df) > 0:
        before = len(df)
        df = df[df['price'] > 0]
        
        price_99 = df['price'].quantile(0.99)
        df = df[df['price'] <= price_99]
        print(f"    Hequr outliers të çmimit: {before - len(df)} rreshta")
    
    if 'mileage' in df.columns and len(df) > 0:
        before = len(df)
        df = df[df['mileage'] >= 0]
        if before - len(df) > 0:
            print(f"    Hequr mileage negative: {before - len(df)} rreshta")
    
    print(f"    {original_rows} → {len(df)} rreshta (hequr {original_rows - len(df)})")
    
    return df

In [4]:
print(" RIEMËRTIMI I SKEDARËVE")


files_to_rename = [
    ('unclean cclass.csv', 'unclean_cclass.csv'),
    ('unclean focus.csv', 'unclean_focus.csv')
]

for old_name, new_name in files_to_rename:
    old_path = DATA_DIR / old_name
    new_path = DATA_DIR / new_name
    if old_path.exists():
        os.rename(old_path, new_path)
        print(f" Riemërtuar: {old_name} → {new_name}")

 RIEMËRTIMI I SKEDARËVE


In [5]:
brands = [
    ('audi', 'audi'),
    ('bmw', 'bmw'),
    ('cclass', 'cclass'),
    ('focus', 'focus'),
    ('ford', 'ford'),
    ('hyundi', 'hyundai'),
    ('merc', 'mercedes'),
    ('skoda', 'skoda'),
    ('toyota', 'toyota'),
    ('vauxhall', 'vauxhall'),
    ('vw', 'volkswagen'),
    ('unclean_cclass', 'cclass'),
    ('unclean_focus', 'focus')
]

print(" BRANDET PËR T'U PASTRUAR:")


for original, display in brands:
    file_path = DATA_DIR / f'{original}.csv'
    exists = "ok" if file_path.exists() else "error in file path"
    print(f"   {exists} {original:20} → {display}")

 BRANDET PËR T'U PASTRUAR:
   ok audi                 → audi
   ok bmw                  → bmw
   ok cclass               → cclass
   ok focus                → focus
   ok ford                 → ford
   ok hyundi               → hyundai
   ok merc                 → mercedes
   ok skoda                → skoda
   ok toyota               → toyota
   ok vauxhall             → vauxhall
   ok vw                   → volkswagen
   ok unclean_cclass       → cclass
   ok unclean_focus        → focus


In [6]:
print(" PASTRIMI I TË DHËNAVE")

results = {}

for original, display in brands:
    input_file = DATA_DIR / f'{original}.csv'
    
    if not input_file.exists():
        print(f"\n {original}.csv nuk u gjet - anashkalohet")
        continue
    
    print(f" Duke përpunuar: {original} → {display}")
    
    
    df = pd.read_csv(input_file)
    original_rows = len(df)
    
    
    df_cleaned = clean_car_dataset(original, display, df)
    
    if original == 'unclean_cclass':
        output_name = 'cclass_cleaned'
    elif original == 'unclean_focus':
        output_name = 'focus_cleaned'
    else:
        output_name = f'{display}_cleaned'
    
   
    output_file = CLEANED_DIR / f'{output_name}.csv'
    df_cleaned.to_csv(output_file, index=False)
    
    
    removed_total = original_rows - len(df_cleaned)
    percentage = (removed_total / original_rows * 100) if original_rows > 0 else 0
    
    report = f"""
    ========================================
    RAPORTI I PASTRIMIT: {original.upper()} → {display.upper()}
    ========================================
    Data: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
    
    STATISTIKAT:
    - Rreshtat para pastrimit: {original_rows}
    - Rreshtat pas pastrimit: {len(df_cleaned)}
    - Rreshtat e hequr: {removed_total}
    - Përqindja e hequr: {percentage:.1f}%
    
    KOLONAT PËRFUNDIMTARE:
    {list(df_cleaned.columns)}
    
    STATISTIKAT NUMERIKE:
    {df_cleaned.describe().to_string() if len(df_cleaned.select_dtypes(include=[np.number]).columns) > 0 else 'Nuk ka kolona numerike'}
    ========================================
    """
    
    report_file = REPORTS_DIR / f'{original}_report.txt'
    with open(report_file, 'w', encoding='ascii', errors='ignore') as f:
        f.write(report)
    
  
    results[original] = {
        'display': display,
        'original_rows': original_rows,
        'cleaned_rows': len(df_cleaned),
        'removed': removed_total,
        'percentage': percentage,
        'output_file': output_name
    }
    
    print(f"\n {original} PËRFUNDOI!")

 PASTRIMI I TË DHËNAVE
 Duke përpunuar: audi → audi

    Pastrimi i audi...
    Konvertimi i kolonës 'price'...
       Tipi i ri: float64
   Hequr vlera null: 0 rreshta
    Hequr outliers të çmimit: 105 rreshta
    10668 → 10460 rreshta (hequr 208)

 audi PËRFUNDOI!
 Duke përpunuar: bmw → bmw

    Pastrimi i bmw...
    Konvertimi i kolonës 'price'...
       Tipi i ri: float64
   Hequr vlera null: 0 rreshta
    Hequr outliers të çmimit: 107 rreshta
    10781 → 10557 rreshta (hequr 224)

 bmw PËRFUNDOI!
 Duke përpunuar: cclass → cclass

    Pastrimi i cclass...
    Konvertimi i kolonës 'price'...
       Tipi i ri: float64
   Hequr vlera null: 0 rreshta
    Hequr outliers të çmimit: 38 rreshta
    3899 → 3759 rreshta (hequr 140)

 cclass PËRFUNDOI!
 Duke përpunuar: focus → focus

    Pastrimi i focus...
    Konvertimi i kolonës 'price'...
       Tipi i ri: float64
   Hequr vlera null: 0 rreshta
    Hequr outliers të çmimit: 48 rreshta
    5454 → 4710 rreshta (hequr 744)

 focus PËRFUNDOI!

In [7]:
print(" PËRMBLEDHJA FINALE")


print("\n STATISTIKAT:")
print(f"   • Brande të pastruar: {len(results)}")
print(f"   • Skedarë të pastruar: {len(list(CLEANED_DIR.glob('*.csv')))}")
print(f"   • Raporte të krijuar: {len(list(REPORTS_DIR.glob('*.txt')))}")

print("\n MAPPING I KRYER:")
print("   • hyundi  → hyundai ✓")
print("   • merc    → mercedes ✓")
print("   • unclean_cclass → cclass ✓")
print("   • unclean_focus → focus ✓")

print("\n SKEDARËT E KRIJUAR:")
print(f"    data/cleaned/     - Të dhënat e pastruara")
for f in CLEANED_DIR.glob('*.csv'):
    print(f"      • {f.name}")
print(f"    data/reports/     - Raportet e pastrimit")
for f in REPORTS_DIR.glob('*.txt'):
    print(f"      • {f.name}")

print("\n REZULTATET PËR ÇDO BRAND:")
for original, data in results.items():
    print(f"   {original:20} → {data['original_rows']:6} → {data['cleaned_rows']:6} rreshta (hequr {data['removed']})")


print(" PASTRIMI PËRFUNDOI!")


 PËRMBLEDHJA FINALE

 STATISTIKAT:
   • Brande të pastruar: 13
   • Skedarë të pastruar: 11
   • Raporte të krijuar: 13

 MAPPING I KRYER:
   • hyundi  → hyundai ✓
   • merc    → mercedes ✓
   • unclean_cclass → cclass ✓
   • unclean_focus → focus ✓

 SKEDARËT E KRIJUAR:
    data/cleaned/     - Të dhënat e pastruara
      • audi_cleaned.csv
      • bmw_cleaned.csv
      • cclass_cleaned.csv
      • focus_cleaned.csv
      • ford_cleaned.csv
      • hyundai_cleaned.csv
      • mercedes_cleaned.csv
      • skoda_cleaned.csv
      • toyota_cleaned.csv
      • vauxhall_cleaned.csv
      • volkswagen_cleaned.csv
    data/reports/     - Raportet e pastrimit
      • audi_report.txt
      • bmw_report.txt
      • cclass_report.txt
      • focus_report.txt
      • ford_report.txt
      • hyundi_report.txt
      • merc_report.txt
      • skoda_report.txt
      • toyota_report.txt
      • unclean_cclass_report.txt
      • unclean_focus_report.txt
      • vauxhall_report.txt
      • vw_report.txt
